# Market Regime Detector
**Using Hidden Markov Models to classify S&P 500 market regimes**

This notebook walks through:
1. Fetching SPX data and engineering features
2. Training a Gaussian HMM with 3 hidden states
3. Labelling states as bull / bear / sideways
4. Backtesting a regime-switching strategy
5. Visualising the regime overlay chart

---

## Step 1 — Install & import dependencies

In [ ]:
# Run this cell once to install dependencies
import sys
!{sys.executable} -m pip install hmmlearn yfinance pandas numpy matplotlib scikit-learn joblib --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import joblib

import yfinance as yf
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('All imports successful ✓')

---
## Step 2 — Data collection & feature engineering

We build three features that historically separate market regimes:

| Feature | What it captures |
|---|---|
| `log_ret` | Daily direction and magnitude of price moves |
| `vol_20` | 20-day rolling volatility — spikes in bear markets |
| `ma_ratio` | 50-day MA / 200-day MA − 1 — trend direction signal |

In [ ]:
TICKER = "^GSPC"   # S&P 500
START  = "2000-01-01"
END    = None      # None = today

def get_features(ticker=TICKER, start=START, end=END):
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    close = df["Close"].squeeze()

    log_ret  = np.log(close / close.shift(1))
    vol_20   = log_ret.rolling(20).std()
    ma_ratio = close.rolling(50).mean() / close.rolling(200).mean() - 1

    features = pd.DataFrame({
        "log_ret" : log_ret,
        "vol_20"  : vol_20,
        "ma_ratio": ma_ratio,
    }).dropna()

    return close.loc[features.index], features

close, features = get_features()

print(f"Data range  : {features.index[0].date()} → {features.index[-1].date()}")
print(f"Trading days: {len(features):,}")
features.tail()

In [ ]:
# Quick look at feature distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
labels = ["Log return", "20-day volatility", "MA ratio (50/200)"]
colors = ["#378ADD", "#E24B4A", "#1D9E75"]

for ax, col, label, color in zip(axes, features.columns, labels, colors):
    features[col].hist(bins=80, ax=ax, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(label, fontsize=11)
    ax.set_ylabel("Frequency")

plt.suptitle("Feature distributions", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Step 3 — Train the Hidden Markov Model

A **Gaussian HMM** assumes each hidden state emits observations from a multivariate Gaussian distribution. The **Baum-Welch** (EM) algorithm iteratively estimates:
- The transition matrix (how likely a regime is to persist or switch)
- The Gaussian mean & covariance for each state

We run `n_fits` random restarts and keep the model with the highest log-likelihood.

In [ ]:
N_STATES = 3    # bull / bear / sideways
N_ITER   = 1000
N_FITS   = 10   # random restarts — keep best

def train(features, n_states=N_STATES, n_iter=N_ITER, n_fits=N_FITS):
    scaler = StandardScaler()
    X = scaler.fit_transform(features.values)

    best_model, best_score = None, -np.inf

    for seed in range(n_fits):
        model = GaussianHMM(
            n_components    = n_states,
            covariance_type = "full",
            n_iter          = n_iter,
            random_state    = seed
        )
        model.fit(X)
        score = model.score(X)
        if score > best_score:
            best_score = score
            best_model = model

    print(f"Best log-likelihood : {best_score:,.1f}")
    print(f"Converged           : {best_model.monitor_.converged}")
    return best_model, scaler

model, scaler = train(features)
print("\nModel trained ✓")

In [ ]:
# Inspect learned state parameters (un-scaled for readability)
X_scaled = scaler.transform(features.values)
raw_means = scaler.inverse_transform(model.means_)

state_summary = pd.DataFrame(
    raw_means,
    columns=["mean log_ret", "mean vol_20", "mean ma_ratio"],
    index=[f"State {i}" for i in range(N_STATES)]
)
state_summary["mean log_ret"]  = state_summary["mean log_ret"].map("{:.5f}".format)
state_summary["mean vol_20"]   = state_summary["mean vol_20"].map("{:.5f}".format)
state_summary["mean ma_ratio"] = state_summary["mean ma_ratio"].map("{:.5f}".format)

print("Learned state centroids (original scale):")
state_summary

---
## Step 4 — Label & interpret regimes

hmmlearn assigns arbitrary integer IDs (0, 1, 2) to states. We sort them by their **mean log-return**:
- Lowest mean return → **bear**
- Highest mean return → **bull**
- Middle → **sideways**

In [ ]:
REGIME_COLORS = {"bull": "#1D9E75", "bear": "#E24B4A", "sideways": "#888780"}

def label_regimes(model, scaler, features, close):
    X = scaler.transform(features.values)
    states = model.predict(X)

    mean_rets = {
        s: features["log_ret"].values[states == s].mean()
        for s in range(model.n_components)
    }
    sorted_states = sorted(mean_rets, key=mean_rets.get)  # bear → sideways → bull
    label_map = {
        sorted_states[0]: "bear",
        sorted_states[1]: "sideways",
        sorted_states[2]: "bull",
    }

    df = pd.DataFrame({
        "close" : close,
        "state" : states,
        "regime": pd.Series(states, index=features.index).map(label_map),
    })
    return df

regime_df = label_regimes(model, scaler, features, close)

# Regime distribution
dist = regime_df["regime"].value_counts(normalize=True).mul(100).round(1)
print("Regime distribution (% of trading days):")
print(dist.to_string())

In [ ]:
# Transition matrix — how sticky are the regimes?
trans_df = pd.DataFrame(
    model.transmat_,
    index=["bear", "sideways", "bull"],
    columns=["→ bear", "→ sideways", "→ bull"]
).round(4)

# Re-order rows to match label order
sorted_states = sorted(
    range(N_STATES),
    key=lambda s: features["log_ret"].values[
        model.predict(scaler.transform(features.values)) == s
    ].mean()
)
trans_df = trans_df.iloc[sorted_states]

print("Transition matrix (row = current state, col = next state):")
trans_df

---
## Step 5 — Visualise the regime overlay

In [ ]:
def plot_regimes(df, title="S&P 500 — HMM Regime Overlay"):
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(14, 7),
        gridspec_kw={"height_ratios": [3, 1]},
        sharex=True
    )

    # --- price chart with shaded regimes ---
    ax1.plot(df.index, df["close"], color="#378ADD", lw=0.9, zorder=3)

    prev_regime, start_idx = df["regime"].iloc[0], df.index[0]
    for date, row in df.iterrows():
        if row["regime"] != prev_regime:
            ax1.axvspan(start_idx, date,
                        alpha=0.20, color=REGIME_COLORS[prev_regime], lw=0)
            start_idx = date
            prev_regime = row["regime"]
    ax1.axvspan(start_idx, df.index[-1],
                alpha=0.20, color=REGIME_COLORS[prev_regime], lw=0)

    patches = [mpatches.Patch(color=c, alpha=0.6, label=r)
               for r, c in REGIME_COLORS.items()]
    ax1.legend(handles=patches, loc="upper left", fontsize=9, framealpha=0)
    ax1.set_ylabel("SPX price", fontsize=10)
    ax1.set_title(title, fontsize=13, pad=10)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    # --- regime strip ---
    regime_num = df["regime"].map({"bear": 0, "sideways": 1, "bull": 2})
    colors_strip = df["regime"].map(REGIME_COLORS)
    ax2.bar(df.index, [1]*len(df), color=colors_strip, width=2, align="center")
    ax2.set_yticks([0.5])
    ax2.set_yticklabels(["Regime"], fontsize=9)
    ax2.set_ylim(0, 1)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax2.xaxis.set_major_locator(mdates.YearLocator(2))

    plt.tight_layout()
    plt.show()

plot_regimes(regime_df)

---
## Step 6 — Backtest: regime-switching strategy

**Strategy:** Hold SPX in bull regimes, move to cash otherwise.  
**Evaluation:** Compare annualised Sharpe ratio and max drawdown vs buy-and-hold.

> ⚠️ **Note:** This is an in-sample backtest. Always validate on a held-out period (e.g. train 2000–2018, test 2019–present).

In [ ]:
def backtest(df):
    log_ret = np.log(df["close"] / df["close"].shift(1)).dropna()
    df = df.loc[log_ret.index]

    strat = log_ret.copy()
    strat[df["regime"] != "bull"] = 0.0  # cash outside bull

    def sharpe(r):
        return (r.mean() / r.std()) * np.sqrt(252)

    def max_dd(r):
        cum = (1 + r).cumprod()
        return (cum / cum.cummax() - 1).min()

    def cagr(r):
        n_years = len(r) / 252
        return (1 + r).prod() ** (1 / n_years) - 1

    results = pd.DataFrame({
        "Strategy (regime-switching)": [
            f"{sharpe(strat):.2f}",
            f"{max_dd(strat)*100:.1f}%",
            f"{cagr(strat)*100:.1f}%",
        ],
        "Buy and hold": [
            f"{sharpe(log_ret):.2f}",
            f"{max_dd(log_ret)*100:.1f}%",
            f"{cagr(log_ret)*100:.1f}%",
        ],
    }, index=["Sharpe ratio (ann.)", "Max drawdown", "CAGR"])

    return results, log_ret, strat

results, log_ret, strat = backtest(regime_df)
results

In [ ]:
# Equity curve comparison
cum_strat = (1 + strat).cumprod()
cum_bah   = (1 + log_ret).cumprod()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(cum_strat.index, cum_strat.values, color="#1D9E75", lw=1.2, label="Regime-switching strategy")
ax.plot(cum_bah.index,   cum_bah.values,   color="#378ADD", lw=1.0, label="Buy and hold", alpha=0.8)
ax.set_title("Cumulative returns: strategy vs buy-and-hold", fontsize=13)
ax.set_ylabel("Growth of $1", fontsize=10)
ax.legend(fontsize=10, framealpha=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:.1f}"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
plt.tight_layout()
plt.show()

---
## Step 7 — Save the model

Save the trained model and scaler so you can load them in a Streamlit app or API without retraining.

In [ ]:
joblib.dump({"model": model, "scaler": scaler}, "regime_model.pkl")
print("Model saved to regime_model.pkl ✓")

# To reload:
# saved = joblib.load("regime_model.pkl")
# model, scaler = saved["model"], saved["scaler"]

---
## Next steps

- **Train/test split** — retrain on 2000–2018 only, evaluate on 2019–present to test out-of-sample performance
- **More features** — add RSI, VIX level, credit spreads, or breadth indicators
- **More regimes** — try `n_components=4` to split bull into *trending* vs *low-vol grind*
- **Deploy** — wrap `model.predict()` in a Streamlit app with `st.plotly_chart()` for an interactive live dashboard
- **Interview prep** — be ready to explain the Baum-Welch EM algorithm and why you chose `covariance_type="full"`